In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributions as dist
import numpy as np
from torch.utils.tensorboard import SummaryWriter 
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

import wandb

from raw import load_imdb_synth, load_xor

In [2]:


"""
Assignment 3B: Questions 10–14 — Autoregressive Transformer
============================================================
Deep dive: every design decision is explained, with mathematics and alternatives.

OVERVIEW OF WHAT WE ARE BUILDING
----------------------------------
A GPT-style autoregressive language model. Given a sequence of tokens
  x_1, x_2, ..., x_T
the model is trained to predict the next token at every position:
  p(x_{t+1} | x_1, ..., x_t)   for t = 1..T

This is done by running the whole sequence through a *causal* (masked) transformer
and reading off the logits at every time step simultaneously — making training
O(T) forward passes worth of information from a single pass. That efficiency is
the whole point of the autoregressive formulation.


"""

# ─────────────────────────────────────────────────────────────────────────────
# QUESTION 10 — Batch sampler for autoregressive training
# ─────────────────────────────────────────────────────────────────────────────
"""
QUESTION 10
===========
Build a function that takes a dataset (a single long integer tensor), and
returns a random batch of shape (b, L+1).

WHY A SINGLE LONG SEQUENCE?
----------------------------
Unlike classification, where each example is a separate sentence, language
modelling treats the entire corpus as ONE continuous stream of tokens.
We then cut out random windows of length L+1:

    corpus:  [t0, t1, t2, t3, ... tN]
              └──────────┘  <- one window of length L+1
                     └──────────┘  <- another window

From each window we split:
    input  = window[:-1]   (length L)
    target = window[1:]    (length L)

This is the *shifted* prediction target: at position i, predict position i+1.

MATHEMATICAL VIEW
-----------------
We are maximising the log-likelihood of the data under the model:

    L = Σ_t  log p_θ(x_{t+1} | x_1, ..., x_t)

Slicing random windows is an unbiased stochastic estimate of this sum.

ALTERNATIVES
------------
1. Sequential (non-random) batching: iterate through corpus in order.
   - Pro: every token seen exactly once per epoch, no overlap waste.
   - Con: consecutive batches are correlated → noisier gradients in practice.
2. Document-aware batching: respect sentence/document boundaries.
   - Used for fine-tuning; adds a [SEP] or <eos> token at boundaries.
3. Pack-and-pad: pad shorter documents, group by similar lengths.
   - Better for classification; wasteful for LM because we can always find a
     length-L window without crossing a boundary.
"""

def sample_batch(data: torch.Tensor, b: int, l: int) -> torch.Tensor:
    """
    Sample a batch of b random windows of length l+1 from `data`.

    Args:
        data:  1-D integer tensor of shape (N,)  — the whole corpus.
        b:     batch size
        l:     context length (the model sees l tokens, predicts l tokens)

    Returns:
        Tensor of shape (b, l+1).  Slice [:, :-1] = input, [:, 1:] = target.

    Implementation detail — why torch.no_grad() / detach()?
    --------------------------------------------------------
    We must NOT let this sampling become part of the autograd computation graph.
    The integer indices are not differentiable anyway, but wrapping in
    torch.no_grad() makes our intent explicit and avoids any accidental
    graph node creation from surrounding code.
    """
    N = data.size(0)
    # Draw b random starting indices in [0, N - l - 1]
    # torch.randint is exclusive on the high end, so high = N - l
    starts = torch.randint(low=0, high=N - l, size=(b,))   # shape (b,)

    # Vectorised slice: build index tensor [starts, starts+1, ..., starts+l]
    # arange gives [0, 1, ..., l], broadcast-add starts[:,None] gives (b, l+1)
    offsets = torch.arange(l + 1, device=data.device)      # shape (l+1,)
    indices = starts[:, None] + offsets[None, :]            # shape (b, l+1)

    # Index into data; .detach() prevents gradient flow (data is discrete anyway)
    return data[indices].detach()

In [10]:
example_data = torch.arange(10) 
b = 2
l = 6
print("example_data =", example_data)
print("b =", b)
print("l =", l)

example_data = tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
b = 2
l = 6


In [9]:
N = example_data.size(0)
print("N =", N)

N = 10


In [11]:
starts = torch.randint(low=0, high=N - l, size=(b,))
print("starts =", starts)

starts = tensor([2, 2])


In [12]:
ofsets = torch.arange(l + 1)      # shape (l+1,)
print("ofsets =", ofsets)

ofsets = tensor([0, 1, 2, 3, 4, 5, 6])


In [13]:
indices = starts[:, None] + ofsets[None, :]            # shape (b, l+1)
print("indices =", indices)

indices = tensor([[2, 3, 4, 5, 6, 7, 8],
        [2, 3, 4, 5, 6, 7, 8]])
